# 02 Pilot Local LLMs

Objective: run a bounded pilot against one local OpenAI-compatible model and check JSON parse rate before the full experiment.

By default this notebook calls the configured model. Set `RUN_PILOT=false` to skip requests.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
DATASET_ID = eu.normalize_dataset_id(os.getenv("DATASET_ID", "nice"))
DATASET_SUFFIX = eu.dataset_suffix(DATASET_ID)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)
ARTIFACT_SUFFIX = eu.dataset_variant_suffix(DATASET_ID, BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, DATASET_ID, BENCHMARK_VARIANT


## Configure Local Endpoint


In [ ]:
HOST = os.getenv("HOST", CONFIG["llm"]["host"])
configured_models = [m.strip() for m in os.getenv("MODELS", ",".join(CONFIG["llm"]["models"])).split(",") if m.strip()]
MODEL = os.getenv("MODEL", configured_models[0])
RUN_PILOT = os.getenv("RUN_PILOT", "true").lower() in {"1", "true", "yes"}
REQUEST_CONCURRENCY = eu.resolve_llm_concurrency(CONFIG)

deterministic = CONFIG["llm"]["deterministic"]
stochastic = CONFIG["llm"]["stochastic"]
print({
    "HOST": HOST,
    "MODEL": MODEL,
    "RUN_PILOT": RUN_PILOT,
    "REQUEST_CONCURRENCY": REQUEST_CONCURRENCY,
    "BENCHMARK_VARIANT": BENCHMARK_VARIANT,
})


## Select Pilot Items


In [ ]:
benchmark_path = eu.artifact_path(PROJECT_ROOT / "data/processed/benchmark_items.csv", DATASET_ID, BENCHMARK_VARIANT)
benchmark = eu.read_csv_rows(benchmark_path)
pilot_seed_count = int(CONFIG["project"]["pilot_seed_count"])
pilot_seed_ids = sorted({row["seed_id"] for row in benchmark})[:pilot_seed_count]
pilot_items = [row for row in benchmark if row["seed_id"] in pilot_seed_ids]
planned_calls = len(pilot_items) * 2 * (1 + int(stochastic["samples"]))
print(f"Benchmark path: {benchmark_path}")
print(f"Pilot items: {len(pilot_items)} ({pilot_seed_count} seeds)")
print(f"Planned calls for one model across both tasks: {planned_calls}")


## Run Pilot


In [ ]:
task1_template = eu.load_prompt(PROJECT_ROOT / "prompts/mandatory_entailment.txt")
task2_template = eu.load_prompt(PROJECT_ROOT / "prompts/modality_extraction.txt")

def prompt_for(task, item):
    if task == "task1":
        return eu.render_prompt(
            task1_template,
            source_statement=item["source_statement"],
            candidate_requirement=item["candidate_requirement"],
        )
    if task == "task2":
        return eu.render_prompt(task2_template, source_statement=item["source_statement"])
    raise ValueError(task)

def request_job(
    item,
    task,
    model,
    sample_kind,
    sample_index,
    temperature,
    top_p,
    run_id,
    request_index,
    prompt=None,
    prompt_version=None,
):
    prompt = prompt if prompt is not None else prompt_for(task, item)
    return {
        "request_index": request_index,
        "run_id": run_id,
        "model": model,
        "host": HOST,
        "task": task,
        "item": item,
        "sample_index": sample_index,
        "sample_kind": sample_kind,
        "temperature": temperature,
        "top_p": top_p,
        "prompt_version": prompt_version or CONFIG["project"]["prompt_version"],
        "prompt": prompt,
        "max_tokens": int(CONFIG["llm"]["max_tokens"]),
        "timeout_s": int(CONFIG["llm"]["timeout_s"]),
        "api_key_env": CONFIG["llm"]["api_key_env"],
    }


In [ ]:
output_path = eu.artifact_path(PROJECT_ROOT / "data/processed/model_outputs_raw_pilot.jsonl", DATASET_ID, BENCHMARK_VARIANT)
run_id = eu.new_run_id("pilot" if BENCHMARK_VARIANT == "must" else f"pilot-{BENCHMARK_VARIANT}")
records = []
jobs = []

for item in pilot_items:
    for task in ["task1", "task2"]:
        jobs.append(request_job(
            item=item,
            task=task,
            model=MODEL,
            sample_kind="deterministic",
            sample_index=0,
            temperature=float(deterministic["temperature"]),
            top_p=float(deterministic["top_p"]),
            run_id=run_id,
            request_index=len(jobs),
        ))
        for sample_index in range(int(stochastic["samples"])):
            jobs.append(request_job(
                item=item,
                task=task,
                model=MODEL,
                sample_kind="stochastic",
                sample_index=sample_index,
                temperature=float(stochastic["temperature"]),
                top_p=float(stochastic["top_p"]),
                run_id=run_id,
                request_index=len(jobs),
            ))
assert len(jobs) == planned_calls

if RUN_PILOT:
    print(f"Dispatching {len(jobs)} pilot calls with concurrency={REQUEST_CONCURRENCY}")
    for record in eu.run_completion_jobs(jobs, max_workers=REQUEST_CONCURRENCY):
        eu.append_jsonl(output_path, record)
        records.append(record)
        if len(records) % 25 == 0 or len(records) == len(jobs):
            print(f"Completed {len(records)}/{len(jobs)} pilot calls")
    print(f"Wrote {len(records)} pilot records to {output_path}")
else:
    print("Pilot not run. Set RUN_PILOT=true in the environment or edit RUN_PILOT to True.")


## Pilot Gate


In [ ]:
pilot_rows = [row for row in eu.read_jsonl(output_path) if row.get("run_id") == run_id] if RUN_PILOT else []
if pilot_rows:
    ok = sum(1 for row in pilot_rows if row["parse_status"] == "ok")
    parse_rate = ok / len(pilot_rows)
    avg_latency = sum(float(row["latency_s"] or 0) for row in pilot_rows) / len(pilot_rows)
    print(f"Parse success: {ok}/{len(pilot_rows)} = {parse_rate:.3f}")
    print(f"Average latency: {avg_latency:.2f}s")
    if parse_rate < 0.95:
        print("Gate failed: inspect invalid outputs before the full run.")
    else:
        print("Gate passed: parse success is >= 95%.")


## Survey-Aligned UQ Pilot Diagnostics


In [ ]:
if pilot_rows:
    pilot_scores = eu.build_uq_scores(benchmark, pilot_rows)
    fields = [
        "model",
        "task",
        "uq_method",
        "source_modality",
        "p_yes",
        "confidence",
        "uncertainty_score",
        "uncertainty_measure",
        "valid_n",
        "total_n",
    ]
    diagnostic_rows = [
        row for row in pilot_scores
        if row["uq_method"] in {"label_self_consistency", "modality_consistency", "predictive_entropy", "variation_ratio"}
    ]
    print(eu.markdown_table(diagnostic_rows[:24], fields))
else:
    print("No pilot rows available. Run the pilot to inspect entropy and variation-ratio diagnostics.")


## Optional Logprob Capability Probe


In [ ]:
RUN_LOGPROB_PROBE = os.getenv("RUN_LOGPROB_PROBE", "true").lower() in {"1", "true", "yes"}
logprob_probe_path = eu.artifact_path(PROJECT_ROOT / "outputs/logprob_probe.json", DATASET_ID, BENCHMARK_VARIANT)

if RUN_LOGPROB_PROBE:
    probe = eu.logprob_support_probe(
        host=HOST,
        model=MODEL,
        api_key_env=CONFIG["llm"]["api_key_env"],
        timeout_s=int(CONFIG["llm"]["timeout_s"]),
    )
    eu.write_json(logprob_probe_path, probe)
    print(probe)
else:
    print("Logprob probe not run. Set RUN_LOGPROB_PROBE=true to test token-level UQ support via /v1/responses.")
    print(f"Probe output path when enabled: {logprob_probe_path}")


## Prompt Sensitivity Check


In [ ]:
strict_template = eu.load_prompt(PROJECT_ROOT / "prompts/mandatory_entailment_strict.txt")
sensitivity_raw_path = eu.artifact_path(PROJECT_ROOT / "data/processed/model_outputs_raw_prompt_sensitivity.jsonl", DATASET_ID, BENCHMARK_VARIANT)
sensitivity_summary_path = eu.artifact_path(PROJECT_ROOT / "outputs/prompt_sensitivity_summary.csv", DATASET_ID, BENCHMARK_VARIANT)
RUN_PROMPT_SENSITIVITY = os.getenv("RUN_PROMPT_SENSITIVITY", "true").lower() in {"1", "true", "yes"}

sensitivity_records = []
sensitivity_run_id = eu.new_run_id("prompt-sensitivity" if BENCHMARK_VARIANT == "must" else f"prompt-sensitivity-{BENCHMARK_VARIANT}")
task1_pilot_items = list(pilot_items)
sensitivity_jobs = []

if RUN_PROMPT_SENSITIVITY:
    for prompt_name, template in [("default", task1_template), ("strict", strict_template)]:
        for item in task1_pilot_items:
            prompt = eu.render_prompt(
                template,
                source_statement=item["source_statement"],
                candidate_requirement=item["candidate_requirement"],
            )
            sensitivity_jobs.append(request_job(
                run_id=f"{sensitivity_run_id}-{prompt_name}",
                model=f"{MODEL}:{prompt_name}",
                task="task1",
                item=item,
                sample_index=0,
                sample_kind="deterministic",
                temperature=float(deterministic["temperature"]),
                top_p=float(deterministic["top_p"]),
                prompt_version=f"{CONFIG['project']['prompt_version']}:{prompt_name}",
                prompt=prompt,
                request_index=len(sensitivity_jobs),
            ))
    print(f"Dispatching {len(sensitivity_jobs)} prompt-sensitivity calls with concurrency={REQUEST_CONCURRENCY}")
    for record in eu.run_completion_jobs(sensitivity_jobs, max_workers=REQUEST_CONCURRENCY):
        eu.append_jsonl(sensitivity_raw_path, record)
        sensitivity_records.append(record)
        if len(sensitivity_records) % 25 == 0 or len(sensitivity_records) == len(sensitivity_jobs):
            print(f"Completed {len(sensitivity_records)}/{len(sensitivity_jobs)} prompt-sensitivity calls")
    print(f"Wrote {len(sensitivity_records)} prompt-sensitivity records to {sensitivity_raw_path}")
else:
    print("Prompt sensitivity not run. Set RUN_PROMPT_SENSITIVITY=true or RUN_PILOT=true.")

sensitivity_summary = eu.prompt_sensitivity_summary(benchmark, sensitivity_records)
eu.write_csv_rows(
    sensitivity_summary_path,
    sensitivity_summary,
    fieldnames=["model", "prompt_run_id", "n", "accuracy", "weak_source_high_p_yes_80", "weak_source_high_p_yes_90", "mean_weak_p_yes"],
)
print(f"Wrote prompt sensitivity summary: {sensitivity_summary_path}")
print(eu.markdown_table(sensitivity_summary, ["model", "prompt_run_id", "n", "accuracy", "weak_source_high_p_yes_80", "weak_source_high_p_yes_90", "mean_weak_p_yes"]))


## Task 2 Modality Prompt Validity Check


In [ ]:
task2_labels_only_template = eu.load_prompt(PROJECT_ROOT / "prompts/modality_extraction_labels_only.txt")
task2_sensitivity_raw_path = eu.artifact_path(PROJECT_ROOT / "data/processed/model_outputs_raw_task2_prompt_sensitivity.jsonl", DATASET_ID, BENCHMARK_VARIANT)
task2_sensitivity_summary_path = eu.artifact_path(PROJECT_ROOT / "outputs/task2_prompt_sensitivity_summary.csv", DATASET_ID, BENCHMARK_VARIANT)
RUN_TASK2_PROMPT_SENSITIVITY = os.getenv("RUN_TASK2_PROMPT_SENSITIVITY", "true").lower() in {"1", "true", "yes"}

task2_sensitivity_items = [row for row in pilot_items if row["source_modality"] == "nice_to_have"]
task2_sensitivity_records = []
task2_sensitivity_run_id = eu.new_run_id("task2-prompt-sensitivity" if BENCHMARK_VARIANT == "must" else f"task2-prompt-sensitivity-{BENCHMARK_VARIANT}")
task2_sensitivity_jobs = []

if RUN_TASK2_PROMPT_SENSITIVITY:
    for prompt_name, template in [("default", task2_template), ("labels_only", task2_labels_only_template)]:
        for item in task2_sensitivity_items:
            prompt = eu.render_prompt(template, source_statement=item["source_statement"])
            task2_sensitivity_jobs.append(request_job(
                run_id=f"{task2_sensitivity_run_id}-{prompt_name}",
                model=f"{MODEL}:task2_{prompt_name}",
                task="task2",
                item=item,
                sample_index=0,
                sample_kind="deterministic",
                temperature=float(deterministic["temperature"]),
                top_p=float(deterministic["top_p"]),
                prompt_version=f"{CONFIG['project']['prompt_version']}:task2_{prompt_name}",
                prompt=prompt,
                request_index=len(task2_sensitivity_jobs),
            ))
    print(f"Dispatching {len(task2_sensitivity_jobs)} Task 2 prompt-validity calls with concurrency={REQUEST_CONCURRENCY}")
    for record in eu.run_completion_jobs(task2_sensitivity_jobs, max_workers=REQUEST_CONCURRENCY):
        eu.append_jsonl(task2_sensitivity_raw_path, record)
        task2_sensitivity_records.append(record)
        if len(task2_sensitivity_records) % 10 == 0 or len(task2_sensitivity_records) == len(task2_sensitivity_jobs):
            print(f"Completed {len(task2_sensitivity_records)}/{len(task2_sensitivity_jobs)} Task 2 prompt-validity calls")
    print(f"Wrote {len(task2_sensitivity_records)} Task 2 prompt-validity records to {task2_sensitivity_raw_path}")
else:
    print("Task 2 prompt-validity check not run. Set RUN_TASK2_PROMPT_SENSITIVITY=true or edit the flag to True.")

task2_sensitivity_summary = eu.task2_prompt_sensitivity_summary(benchmark, task2_sensitivity_records)
task2_sensitivity_fields = [
    "model",
    "prompt_run_id",
    "n",
    "valid_n",
    "parse_success_rate",
    "accuracy",
    "nice_to_have_n",
    "nice_to_have_accuracy",
    "nice_to_have_to_recommended_rate",
    "over_commitment",
    "high_conf_overcommit_80",
    "high_conf_overcommit_90",
]
eu.write_csv_rows(task2_sensitivity_summary_path, task2_sensitivity_summary, fieldnames=task2_sensitivity_fields)
print(f"Wrote Task 2 prompt-validity summary: {task2_sensitivity_summary_path}")
print(eu.markdown_table(task2_sensitivity_summary, task2_sensitivity_fields))
